# B：ABI3 ROI 与面积统计逐格验收

本版独立于已通过的ABI2。原灰度直接进DMA，PL完成可选反相、严格阈值、ROI并集与17个ROI前景计数。
所有代码格初始无输出。先False检查文件，准备实际操作时再改True。
面积是阈值前景数，不能据此宣称SAD/hcorr/cd40和全部缺陷分类已在PL实现。


In [ ]:
from pathlib import Path
import sys
import os
import json
import platform
from datetime import datetime, timezone
ROOT = Path('/home/xilinx/jupyter_notebooks/pcb_b_roi_v03')
RUN_ON_BOARD = False
BITFILE = ROOT / 'release' / 'roi_v0_3' / 'pcb_roi.bit'
PREFLIGHT_OK = False
HARDWARE_PREFLIGHT_OK = False
DRIVER_READY = False
board_result = None
assert (ROOT / 'software' / 'roi_driver.py').is_file(), '请修改ROOT为完整新包目录'
sys.path.insert(0, str(ROOT / 'software'))
print('工程目录：', ROOT)
print('当前系统：', platform.system(), platform.machine())


## 软件核对

只核对冻结输入、两版ROI、标准mask和面积。预期8个案例，latest17白像素326860，
frozen16白像素314801，frozen16与A原mask差异0。这里不下载FPGA。


In [ ]:
PREFLIGHT_OK = False
from roi_reference import audit, sha256
from board_validation import validate_board
offline = audit(ROOT)
print(json.dumps(offline, ensure_ascii=False, indent=2))
assert offline['frozen16_vs_original_A']['match']
assert len(offline['cases']) == 8
PREFLIGHT_OK = True


## 硬件预检

准备实际板上验证时，把第1格RUN_ON_BOARD改True，重新执行第1格→第2格，再到这里。
使用新的独立会话，结束其他占用同一DMA/Overlay的实验。下方预检不下载bit；第4代码格才下载。
PYNQ的XRT环境要在导入pynq之前就存在；本板官方配置值为/usr。


In [ ]:
HARDWARE_PREFLIGHT_OK = False
assert RUN_ON_BOARD is True and PREFLIGHT_OK is True, '先改True并重跑第1、2格'
assert platform.system() == 'Linux' and platform.machine() in ('armv7l', 'armv8l')
assert os.geteuid() == 0, '加载FPGA/MMIO需要板卡管理员运行环境'
if 'pynq' in sys.modules and (not os.environ.get('XILINX_XRT')):
    raise RuntimeError('已有PYNQ导入却缺少XRT环境；先确认硬件停止后使用正确环境的新会话')
os.environ.setdefault('XILINX_XRT', '/usr')
import pynq
import numpy as np
print('PYNQ：', pynq.__version__, 'NumPy：', np.__version__)
release = ROOT / 'release' / 'roi_v0_3'
hashes = json.loads((release / 'sha256.json').read_text(encoding='utf-8'))
for name in ('pcb_roi.bit', 'pcb_roi.hwh'):
    assert sha256(release / name) == hashes[name], 'bit/hwh摘要不匹配：' + name
print('配套bit/hwh校验通过')
HARDWARE_PREFLIGHT_OK = True


## 加载新版本

这格替换当前FPGA配置，先核对同名HWH中DMA与两组GPIO的实例、方向、参数和地址，下载后再读ABI。
新版ctrl/cfg GPIO地址可能与旧版互换，驱动按HWH实例名解析，不按旧地址操作。
本格只运行一次。已有driver时应保留当前状态，不能删除driver或检查语句绕过。


In [ ]:
DRIVER_READY = False
assert RUN_ON_BOARD is True and HARDWARE_PREFLIGHT_OK is True
assert 'driver' not in globals(), '已有driver对象；先确认旧硬件会话安全结束'
from roi_driver import RoiDriver
driver = RoiDriver.from_bitfile(BITFILE, timeout=10.0)
print('ABI3状态：', driver.read_status())
DRIVER_READY = True


## 八案例真实DMA

按顺序执行：32B旁路、64B旁路、8×4重叠ROI/边界、640×480旁路、整图旁路、整图反相阈值、
最新17ROI整图、冻结16ROI整图。每例比较像素、所有17个计数槽、输出前景总数、有效像素数与包计数。
只需运行这个代码格一次，程序自动依次执行，第一处失败即停止。

若出现PoisonedError或无法确认DMA停止，保留driver/缓冲区/内核，先使硬件受控停止再恢复。
轮询期限不能抢占底层卡住的MMIO访问。单次CPU/驱动耗时用于记录，不据此宣称持续帧率或可靠加速比。


In [ ]:
board_result = None
assert RUN_ON_BOARD is True and DRIVER_READY is True and ('driver' in globals())
run_name = 'board_run_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR = ROOT / 'evidence' / run_name
print('本次结果：', RUN_DIR)
board_result = validate_board(driver, ROOT, RUN_DIR)
print(json.dumps(board_result, ensure_ascii=False, indent=2))
assert board_result['passed'] is True


## 本次结果摘要

只在上格实际成功时继续。每次保存实际BIN、期望/实测计数和驱动状态；最后一例还直接对照A原mask。
passed=true的范围是本轮8例，长时稳定性和全部分类能力另行验收。


In [ ]:
assert isinstance(board_result, dict) and board_result.get('passed') is True
assert board_result.get('state') == 'complete'
assert board_result.get('run_directory') == str(RUN_DIR.resolve())
assert len(board_result['records']) == 8
for record in board_result['records']:
    print(record['case'], '像素差异=', record['pixel_check']['mismatches'], '面积一致=', all(record['stats_match'].values()), '输出前景=', record['actual_stats']['output_white_pixels'])
print('A原mask差异：', board_result['records'][-1]['original_A_mask_check']['mismatches'])
print('最终状态：', board_result['final_status'])
print('报告：', RUN_DIR / 'board_report.json')
